# SafeStreet: Model Training & Fine-Tuning Pipeline (Google Colab)
### AI Condition Classification (Keras + TensorFlow) & Object Detection (YOLO)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SafeStreet/SafeStreet/blob/main/SafeStreet_Model_Training.ipynb)

This notebook trains the two AI components of the SafeStreet system:
1. **AI Condition Classifier (Keras + TensorFlow)**: Classifies road infrastructure degradation (Zebra crossing wear, missing school signage, and surface defects).
2. **Object Detection (YOLOv8)**: Detects pedestrians, children, two-wheelers, cars, buses, and roadside signs in school zones.

## 1. Install & Verify Dependencies

In [ ]:
!pip install -q tensorflow keras ultralytics opencv-python matplotlib pandas

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Dataset Preparation for Road Condition Classification
We structure the dataset into 3 key condition categories for school zones:
- `0_intact`: Well-marked, high-visibility pedestrian infrastructure
- `1_worn`: Degraded, flaked, or faded paint markings (>50% wear)
- `2_missing`: Total absence of pedestrian markings / unmarked road surface

In [ ]:
# If using real imagery from Google Drive or Roboflow, un-comment below:
# from google.colab import drive
# drive.mount('/content/drive')

# For rapid reproduction, generate synthetic baseline training samples:
import os, cv2, numpy as np

def generate_colab_dataset(base_dir="dataset"):
    classes = ["0_intact", "1_worn", "2_missing"]
    for split, count in [("train", 150), ("val", 40)]:
        for cls in classes:
            p = os.path.join(base_dir, split, cls)
            os.makedirs(p, exist_ok=True)
            for i in range(count):
                img = np.random.randint(45, 65, (128, 128, 3), dtype=np.uint8)
                if cls == "0_intact":
                    for x in range(10, 128, 30):
                        cv2.rectangle(img, (x, 15), (x+18, 115), (235, 235, 240), -1)
                elif cls == "1_worn":
                    for x in range(10, 128, 30):
                        cv2.rectangle(img, (x, 15), (x+18, 115), (140, 140, 140), -1)
                        for _ in range(15):
                            cv2.circle(img, (np.random.randint(x, x+18), np.random.randint(15, 115)), 3, (50, 50, 50), -1)
                cv2.imwrite(os.path.join(p, f"sample_{i:04d}.jpg"), img)

generate_colab_dataset("dataset")
print("Dataset prepared successfully in /content/dataset/")

## 3. Train Keras Condition Classifier (MobileNetV2 / Custom CNN)

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Data Augmentation Pipeline
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomContrast(0.1)
])

# Model Architecture
inputs = keras.Input(shape=(128, 128, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(3, activation='softmax')(x)

model = keras.Model(inputs=inputs, outputs=outputs, name="SafeStreet_Colab_Classifier")
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

# Save trained model
os.makedirs("models", exist_ok=True)
model.save("models/condition_classifier.keras")
print("Model saved to models/condition_classifier.keras")

## 4. Fine-Tuning YOLOv8 on School Zone Objects
We can fine-tune Ultralytics YOLOv8 on custom school zone datasets containing:
- `school_sign`
- `speed_breaker`
- `pedestrian_child`
- `encroaching_twowheeler`

In [ ]:
from ultralytics import YOLO

# Load pre-trained nano model
yolo_model = YOLO('yolov8n.pt')

# Fine-tune on custom dataset (provide data.yaml path if available):
# yolo_model.train(data='school_safety_data.yaml', epochs=30, imgsz=640)

# Export weights:
# yolo_model.export(format='onnx')

## 5. Download Artifacts for SafeStreet Dashboard

In [ ]:
from google.colab import files
# files.download('models/condition_classifier.keras')
print("Ready to download models/condition_classifier.keras and integrate with app.py!")